# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import ComplementaryNeymanShapExplainer, Explainer
from mllm_shap.shap.normalizers import MinMaxNormalizer

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W1111 01:10:08.417000 59551 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create explainer that will make initial call and then explain it using shapley values using Neyman Formula. When `initial_samples` or `initial_fraction` are not provided, it will use default formula of `max(2, ceil(num_splits / (2 * n^2)))`.

In [6]:
explainer = Explainer(model=model, shap_explainer=ComplementaryNeymanShapExplainer(normalizer=MinMaxNormalizer(), num_samples=200))

Create new chat instance and assign it messages. ComplementaryNeymanShapExplainer currently supports only `SystemRolesSetup.SYSTEM_ASSISTANT` mode. It requires at least one assistant turn to be present.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.ASSISTANT)
chat.add_text("Be helpful and concise.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you? Where have been created?")
chat.end_turn()

# Usage

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {"max_new_tokens": 4, "model_config": ModelConfig(text_temperature=0.2, text_top_k=3)}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-11 01:10:13,507 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-11 01:10:14,184 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 9 (up to 511 additional calls)
2025-11-11 01:10:14,207 - mllm_shap.shap.neyman - INFO - Starting initial sampling step with 2 samples per entry in M


Calculating SHAP values:   0%|          | 0/200 [00:00<?, ?it/s]

2025-11-11 01:10:55,504 - mllm_shap.shap.neyman - INFO - Starting Neyman allocation step with 53 remaining samples


Calculating SHAP values:   0%|          | 0/200 [00:00<?, ?it/s]

Let's see final Shap values.

In [9]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,nan
1,<|im_start|>,nan
2,assistant,nan
3,,nan
4,Be,nan
5,helpful,nan
6,and,nan
7,concise,nan
8,.,nan
9,<|im_end|>,nan


# Tests

Presence matrix:

In [10]:
explainer.shap_explainer._M

tensor([[ 2,  2,  3, 13,  9, 16, 15, 18, 23,  2],
        [ 2,  2,  5,  8, 11, 14, 20, 16, 23,  2],
        [ 2,  3,  6, 11, 11, 14, 17, 15, 22,  2],
        [ 2,  2,  4,  7, 12, 13, 21, 17, 23,  2],
        [ 2,  3,  4,  8, 11, 14, 20, 17, 22,  2],
        [ 2,  4,  5, 13, 12, 13, 15, 16, 21,  2],
        [ 2,  2,  4,  7, 10, 15, 21, 17, 23,  2],
        [ 2,  4,  5,  8, 10, 15, 20, 16, 21,  2],
        [ 2,  3,  6,  9, 14, 11, 19, 15, 22,  2]], device='mps:0',
       dtype=torch.int16)

CC matrices:

In [11]:
explainer.shap_explainer._C

tensor([[-0.4844, -0.1797, -0.1094, -1.2031,  0.2852,  0.7969,  1.1406,  2.1562,
          3.8438,  0.4844],
        [-0.4844, -0.4180, -0.5391, -1.0781, -0.1133,  0.3984,  1.2656,  1.7266,
          3.6250,  0.4844],
        [-0.4844, -0.3125, -0.6172,  0.1914,  0.4609,  0.9727,  2.5312,  1.6484,
          3.7188,  0.4844],
        [-0.4844, -0.2188, -0.2188, -0.2500, -0.4141,  0.0977,  2.0938,  2.0625,
          3.8125,  0.4844],
        [-0.4844, -0.5625, -0.5859, -0.6719, -0.3750,  0.1367,  1.6875,  1.6797,
          3.4688,  0.4844],
        [-0.4844, -0.9336, -0.5586, -1.1250, -0.5820, -0.0703,  1.2188,  1.7188,
          3.0938,  0.4844],
        [-0.4844, -0.2656, -0.4336, -0.5742, -0.6016, -0.0898,  1.7656,  1.8203,
          3.7500,  0.4844],
        [-0.4844, -0.5898, -0.5547, -0.9414, -0.3203,  0.1914,  1.3984,  1.7188,
          3.4062,  0.4844],
        [-0.4844, -0.5781, -0.9141, -1.3750, -0.3867,  0.1250,  0.9688,  1.3516,
          3.4375,  0.4844]], device='mps:0', dt

In [15]:
explainer.shap_explainer._ComplementaryNeymanShapExplainer__C_squared

tensor([[0.1177, 0.0168, 0.0396, 0.3242, 0.0737, 0.2432, 0.2207, 0.3145, 0.7109,
         0.1177],
        [0.1177, 0.0879, 0.0796, 0.2363, 0.1045, 0.2129, 0.3086, 0.2695, 0.6406,
         0.1177],
        [0.1177, 0.0342, 0.0752, 0.1011, 0.1128, 0.2041, 0.4453, 0.2754, 0.6953,
         0.1177],
        [0.1177, 0.0239, 0.0513, 0.0552, 0.1963, 0.1211, 0.4902, 0.3027, 0.7031,
         0.1177],
        [0.1177, 0.1147, 0.0947, 0.1650, 0.0830, 0.2334, 0.3809, 0.2559, 0.6172,
         0.1177],
        [0.1177, 0.2188, 0.0752, 0.2402, 0.1162, 0.2002, 0.3047, 0.2734, 0.5156,
         0.1177],
        [0.1177, 0.0356, 0.0488, 0.1133, 0.1895, 0.1279, 0.4316, 0.3027, 0.6914,
         0.1177],
        [0.1177, 0.0869, 0.0796, 0.1660, 0.1729, 0.1445, 0.3789, 0.2695, 0.6406,
         0.1177],
        [0.1177, 0.1133, 0.1553, 0.2344, 0.2188, 0.0986, 0.3105, 0.1934, 0.6172,
         0.1177]], device='mps:0', dtype=torch.bfloat16)